# 03 — Iterative Debugging Loop & Execution Reward Design
**Goal**: Implement the multi-turn agentic debugging loop ($K=3$ turns) with error feedback (Verification Check V2), define execution-guided reward functions, synthetic bug injection, and collect trajectory preference pairs for DPO.

---

## Step 1: Environment & Module Setup

In [ ]:
import sys, os, shutil

# 1. Clear any previously cached 'src' modules from Python memory
for mod_name in list(sys.modules.keys()):
    if mod_name == "src" or mod_name.startswith("src."):
        del sys.modules[mod_name]

# 2. Prepare working copy of 'src' in /kaggle/working
input_src = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "src" in dirs:
        input_src = os.path.join(root, "src")
        break

working_src = "/kaggle/working/src"
if input_src:
    if os.path.exists(working_src):
        shutil.rmtree(working_src)
    shutil.copytree(input_src, working_src)
    print(f"SUCCESS: Prepared working copy of 'src' at {working_src}")

# 3. Patch models/loader.py with dual torchao override right before get_peft_model
loader_file = os.path.join(working_src, "models", "loader.py")
loader_clean_code = '''"""Model loading utilities with quantization and LoRA configuration."""

import torch
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

DEFAULT_MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"


def load_model_and_tokenizer(
    model_name: str = DEFAULT_MODEL_NAME,
    load_in_4bit: bool = True,
    lora_r: int = 16,
    lora_alpha: int = 32,
    lora_dropout: float = 0.05,
    attach_lora: bool = True,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    try:
        import bitsandbytes
        has_bnb = True
    except (ImportError, Exception):
        has_bnb = False

    if load_in_4bit and has_bnb and torch.cuda.is_available():
        try:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            device_map = "auto"
        except Exception:
            bnb_config = None
            device_map = "auto" if torch.cuda.is_available() else None
    else:
        bnb_config = None
        device_map = "auto" if torch.cuda.is_available() else None

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map=device_map,
        trust_remote_code=True,
    )

    if attach_lora:
        try:
            import peft.import_utils
            peft.import_utils.is_torchao_available = lambda: False
        except Exception:
            pass
        try:
            import peft.tuners.lora.torchao
            peft.tuners.lora.torchao.is_torchao_available = lambda: False
        except Exception:
            pass

        lora_config = LoraConfig(
            r=lora_r,
            lora_alpha=lora_alpha,
            target_modules=["q_proj", "v_proj"],
            lora_dropout=lora_dropout,
            task_type=TaskType.CAUSAL_LM,
        )
        model = get_peft_model(model, lora_config)

    return model, tokenizer
'''
with open(loader_file, "w", encoding="utf-8") as f:
    f.write(loader_clean_code)

# 4. Patch training/dpo.py to prevent top-level TRL import error
dpo_file = os.path.join(working_src, "training", "dpo.py")
dpo_clean_code = '''"""DPO training entry points."""

from datasets import Dataset
from src.debugging.debug_loop import agentic_debug_loop


def make_preference_pairs(problems, model, tokenizer, K: int = 3) -> Dataset:
    """Collect (chosen, rejected) preference pairs from execution debug loop rollouts."""
    pairs = []
    for prob in problems:
        question = prob.get("question", prob.get("prompt", ""))
        test_cases = prob.get("test_cases", [])
        history = agentic_debug_loop(model, tokenizer, question, test_cases, K=K)

        ac_turns = [h for h in history if h["result"]["status"] == "AC"]
        bad_turns = [h for h in history if h["result"]["status"] in ("CE", "RE", "WA", "TLE", "MLE")]

        if ac_turns and bad_turns:
            pairs.append({
                "prompt": question,
                "chosen": ac_turns[0]["code"],
                "rejected": bad_turns[0]["code"],
            })

    return Dataset.from_list(pairs)


def run_dpo_training(model, tokenizer, preference_data, output_dir="./checkpoints/dpo", beta=0.1, learning_rate=5e-5, num_train_epochs=3, per_device_train_batch_size=4):
    try:
        from trl import DPOConfig, DPOTrainer
    except ImportError as e:
        raise ImportError(f"TRL library is required for DPO training: {e}")

    dpo_config = DPOConfig(
        beta=beta,
        learning_rate=learning_rate,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        output_dir=output_dir,
        fp16=True,
        report_to="none",
    )

    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_config,
        train_dataset=preference_data,
        tokenizer=tokenizer,
    )

    dpo_trainer.train()
    dpo_trainer.save_model(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    return dpo_trainer
'''
with open(dpo_file, "w", encoding="utf-8") as f:
    f.write(dpo_clean_code)

# 5. Set /kaggle/working as top priority path
sys.path.insert(0, "/kaggle/working")

# 6. Import all modules
import torch
from datasets import load_dataset
from src.models.loader import load_model_and_tokenizer
from src.debugging.debug_loop import build_prompt, agentic_debug_loop, DebugLoop
from src.rewards.execution_reward import compute_reward, compute_reward_binary, compute_partial_reward, compute_status_aware_reward
from src.training.dpo import make_preference_pairs
from src.error_injection import BugInjector, BugType

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment & modules patched and initialized successfully!")

## Step 2: Synthetic Bug Injection Demo

In [ ]:
injector = BugInjector(seed=42)
correct_code = "def multiply(a, b):\n    return a * b"

print("=== Synthetic Bug Injection ===")
for category in list(BugType):
    res = injector.inject_bug(correct_code, category=category)
    print(f"[{res['bug_type'].upper()}] -> {res['description']}")

## Step 3: Load Model & Multi-Turn Agentic Debugging Loop (Verification Check V2)
Demonstrates the agentic debugging loop up to $K=3$ turns where model inspects its previous attempt and error traceback to fix bugs.

In [ ]:
!pip uninstall -y torchao

import sys, os
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
print(f"Loading model and tokenizer: {MODEL_NAME} in FP16 precision...")
model, tokenizer = load_model_and_tokenizer(
    model_name=MODEL_NAME,
    load_in_4bit=False,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    attach_lora=True,
)

humaneval = load_dataset('openai_humaneval', split='test')
problem = humaneval[0]['prompt']
print("--- HumanEval Problem Statement ---")
print(problem[:250] + "...")

print("\n--- Running Agentic Debugging Loop (K=3) ---")
history = agentic_debug_loop(model, tokenizer, problem, test_cases=[], K=3)

for turn_info in history:
    print(f"\n[Turn {turn_info['turn']}] Status: {turn_info['result']['status']}")
    print(f"Code Snippet:\n{turn_info['code'][:150]}...")
    if turn_info['result'].get('traceback'):
        print(f"Traceback: {turn_info['result']['traceback'][:150]}...")

print("\nVerification Check V2 completed!")

## Step 4: Execution Reward Function Verification
Validates reward scores for AC ($+1.0$), WA/partial pass ($0.0 - 1.0$), and CE/RE ($-0.2$ penalty).

In [ ]:
print("--- Dense Reward Calculations ---")
print(f"AC (5/5 tests): {compute_reward('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward('WA', 3, 5)}")
print(f"CE (0/5 tests): {compute_reward('CE', 0, 5)}")
print(f"RE (0/5 tests): {compute_reward('RE', 0, 5)}")

print("\n--- Binary Reward Calculations (RQ4 Ablation) ---")
print(f"AC (5/5 tests): {compute_reward_binary('AC', 5, 5)}")
print(f"WA (3/5 tests): {compute_reward_binary('WA', 3, 5)}")

## Step 5: Preference Pair Generation for DPO
Collects `(prompt, chosen, rejected)` trajectory pairs from debug rollouts for DPO training.

In [ ]:
print("Generating sample DPO preference pairs...")

eval_data = load_dataset('openai_humaneval', split='test[:5]')
pref_data = make_preference_pairs(eval_data, model, tokenizer, K=3)
print(f"Total preference pairs collected: {len(pref_data)}")
if len(pref_data) > 0:
    print(f"Sample Chosen:\n{pref_data[0]['chosen'][:100]}...")
    print(f"Sample Rejected:\n{pref_data[0]['rejected'][:100]}...")